# HPO LunarLander

Optuna hyperparameter optimization for `VectorTrainer` on LunarLander.

The happy path on Colab is labeled HP1 to HP4.

Hardware: L4 GPU is the intended runtime.

## Set up Colab

In [ ]:
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.colab import setup_colab

COLAB = setup_colab()

## Set up run

In [ ]:
import torch

from hpo.colab import prepare_study_storage
from hpo.evaluation.dashboard import Dashboard
from hpo.lunar_lander.environment import EnvFactory
from hpo.objective import ObjectiveConfig
from hpo.params import HP
from hpo.study import Baseline, StudyRunner
from hpo.notebook_utils import neighbors

ENVIRONMENT_FACTORY = EnvFactory()
STUDY_STORAGE = prepare_study_storage(COLAB)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENVIRONMENT_FACTORY,
    num_envs=16,
    eval_episodes=20,
    device=device,
)
device

## Define baseline

In [ ]:
BASELINE_PARAMS = {
    HP.LEARNING_RATE: 3e-4,
    HP.BATCH_SIZE: 512,
    HP.EPS_END: 0.05,
    HP.EPS_DECAY: 25_000,
    HP.GAMMA: 0.99,
    HP.TAU: 0.005,
    HP.LEARNING_STARTS: 5_000,
    HP.OPTIMIZE_EVERY: 4,
    HP.REPLAY_MEMORY_CAPACITY: 100_000,
    HP.NUM_EPISODES: 600,
}

## Define search spaces

In [ ]:
BATCH_SIZES = [256, 512, 1024]
LEARNING_STARTS = [2_500, 5_000, 10_000]
OPTIMIZE_EVERY = [2, 4, 8]


def suggest_s1_params(trial, incumbent_params):
    trial.suggest_float(HP.LEARNING_RATE, 1e-4, 1e-3, log=True)
    trial.suggest_categorical(HP.BATCH_SIZE, BATCH_SIZES)
    trial.suggest_categorical(HP.LEARNING_STARTS, LEARNING_STARTS)
    trial.suggest_categorical(HP.OPTIMIZE_EVERY, OPTIMIZE_EVERY)


def suggest_s2_params(trial, incumbent_params):
    trial.suggest_float(HP.EPS_END, 0.01, 0.10)
    trial.suggest_int(HP.EPS_DECAY, 10_000, 60_000, log=True)


def suggest_s3_params(trial, incumbent_params):
    trial.suggest_categorical(HP.REPLAY_MEMORY_CAPACITY, [50_000, 100_000, 200_000])


def suggest_s4_params(trial, incumbent_params):
    eps_end = incumbent_params[HP.EPS_END]
    eps_decay = incumbent_params[HP.EPS_DECAY]
    trial.suggest_float(
        HP.LEARNING_RATE,
        incumbent_params[HP.LEARNING_RATE] / 2,
        incumbent_params[HP.LEARNING_RATE] * 2,
        log=True,
    )
    trial.suggest_categorical(
        HP.BATCH_SIZE,
        neighbors(incumbent_params[HP.BATCH_SIZE], BATCH_SIZES),
    )
    trial.suggest_float(
        HP.EPS_END,
        max(0.01, eps_end - 0.02),
        min(0.10, eps_end + 0.02),
    )
    trial.suggest_int(
        HP.EPS_DECAY,
        max(1, eps_decay // 2),
        eps_decay * 2,
        log=True,
    )
    trial.suggest_categorical(
        HP.LEARNING_STARTS,
        neighbors(incumbent_params[HP.LEARNING_STARTS], LEARNING_STARTS),
    )
    trial.suggest_categorical(
        HP.OPTIMIZE_EVERY,
        neighbors(incumbent_params[HP.OPTIMIZE_EVERY], OPTIMIZE_EVERY),
    )


## Run studies

In [ ]:
runner = StudyRunner(
    database_path=STUDY_STORAGE.database_path,
    objective_cfg=OBJECTIVE_CFG,
    baseline=Baseline(params=BASELINE_PARAMS, score=0.0),
    reporter=Dashboard(),
    sync_fn=STUDY_STORAGE.backup,
)

runner.run("s1_update_economy", suggest_s1_params, 40)
runner.run("s2_exploration", suggest_s2_params, 40)
runner.run("s3_replay_capacity", suggest_s3_params, 10)
runner.run("s4_joint_finetune", suggest_s4_params, 30)


## Analyze results

### Reload studies

In [ ]:
# Run only after the runtime environment has been reset.
%pip install -q optuna plotly nbformat

import optuna

def load_study(name):
    return optuna.load_study(
        study_name=name,
        storage=f"sqlite:///{STUDY_STORAGE.database_path(name)}",
    )

study1 = load_study("s1_update_economy")
study2 = load_study("s2_exploration")
study3 = load_study("s3_replay_capacity")
study4 = load_study("s4_joint_finetune")


### Show best hyperparameters

In [ ]:
import pandas as pd

studies = [study1, study2, study3, study4]
columns = ["S1 Update economy", "S2 Exploration", "S3 Replay capacity", "S4 Joint fine-tune"]
table = pd.DataFrame({
    column: study.user_attrs["incumbent_params"]
    for column, study in zip(columns, studies)
})
table.loc["score"] = [study.user_attrs["incumbent_score"] for study in studies]

integer_rows = [HP.BATCH_SIZE, HP.EPS_DECAY, HP.LEARNING_STARTS, HP.OPTIMIZE_EVERY, HP.REPLAY_MEMORY_CAPACITY]
styled_table = table.style.format("{:.3f}")
styled_table = styled_table.format("{:.6f}", subset=pd.IndexSlice[[HP.LEARNING_RATE], :])
styled_table = styled_table.format("{:.0f}", subset=pd.IndexSlice[integer_rows, :])
styled_table = styled_table.format("{:.3f}", subset=pd.IndexSlice[["score"], :])
styled_table = styled_table.set_table_styles([{
    "selector": "tbody tr:last-child th",
    "props": "border-top: 2px solid #888; font-weight: bold",
}])
display(styled_table)


### Plot scores

Score dependence on the hyperparameters optimized in each study.

In [ ]:
from IPython.display import display
from optuna.visualization import plot_contour, plot_slice

print("S1: Update economy")
display(plot_contour(study1, target_name="Score"))
print("S2: Exploration")
display(plot_contour(study2, target_name="Score"))
print("S3: Replay capacity")
display(plot_slice(study3, target_name="Score"))
print("S4: Joint fine-tune")
display(plot_contour(study4, target_name="Score"))
